# Phase 3 / p3_01 -- R1: clean baselines (train + evaluate)

Trains every (model, task, seed) job on **clean** training data, selects the
best epoch by validation macro-F1, evaluates the clean test set AND the full
R2/R6 noise grid in the same in-memory session (see `pipeline/p3_train.py`'s
module docstring for why training and evaluation are one job here, not two
notebooks' worth of separate work), then deletes the checkpoint.

45 jobs total: 5 models x 3 tasks x 3 seeds, `train_variant="clean"`.

**Resumable.** Re-running this notebook (same or a later session) skips any
job already fully present in `results.csv` -- a 12-hour Kaggle cutoff costs
at most one in-flight job.

In [ ]:
# !ls -R /kaggle/input/ | head -30

In [ ]:
# torch is intentionally NOT reinstalled -- Kaggle's GPU image ships a
# CUDA-matched build; see requirements-phase3.txt's top comment.
%pip install -q "transformers>=4.40" "scikit-learn>=1.4" "statsmodels>=0.14" \
    "regex==2024.11.6" \
    "git+https://github.com/csebuetnlp/normalizer@d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9"

import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible -- on Kaggle, enable a GPU accelerator in "
         "Notebook Settings before running a real (non --limit) job.")


In [ ]:
import os
import sys
import shutil
import time
import glob
import pandas as pd

KAGGLE_USER = "mtutul"
DATA_FINAL_ROOT = "/kaggle/input/datasets/mtutul/bangla-noisebench-data"
RESULTS_SRC = "/kaggle/input/datasets/mtutul/bangla-noisebench-results"
RESULTS_DATASET_SLUG = "bangla-noisebench-results"
RESULTS_DIR = "/kaggle/working/results"
CODE_ROOT = "/kaggle/working/repo"
IS_KAGGLE = True

if not os.path.exists(CODE_ROOT):
    rc = os.system("git clone -q https://github.com/TutulMajumder/BanglaNoise_Bench.git " + CODE_ROOT)
    assert rc == 0, "git clone failed -- Internet on?"
sys.path.insert(0, CODE_ROOT)

os.makedirs(RESULTS_DIR, exist_ok=True)

for fname in ("results.csv", "curves.csv"):
    src = os.path.join(RESULTS_SRC, fname)
    if os.path.exists(src) and os.path.getsize(src) > 20:
        shutil.copy(src, os.path.join(RESULTS_DIR, fname))
        print("restored", fname)
    else:
        print("no prior", fname, "-- starting fresh")

dst_preds = os.path.join(RESULTS_DIR, "preds")
os.makedirs(dst_preds, exist_ok=True)
found = glob.glob(os.path.join(RESULTS_SRC, "preds", "**", "*.csv"), recursive=True)
for f in found:
    shutil.copy(f, os.path.join(dst_preds, os.path.basename(f)))
print("restored", len(found), "preds files")

models = set()
for f in os.listdir(dst_preds):
    models.add(f.split("__")[0])
models = sorted(models)

d = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv"))
print(">>> RESUME STATE:", len(d), "rows,", d.run_id.nunique(), "runs,", len(found), "preds")
print(">>> preds models:", models)

assert d.run_id.nunique() >= 45, "RESUME BROKEN -- stop"
assert len(found) >= 88, "PREDS NOT RESTORED -- stop"
assert len(models) == 5, "missing model preds -- stop"

assert os.path.isdir(os.path.join(DATA_FINAL_ROOT, "sentnob")), "sentnob missing"
print("data OK:", sorted(os.listdir(DATA_FINAL_ROOT)))
print("code OK:", "pipeline" in os.listdir(CODE_ROOT))

In [ ]:
from pipeline import p3_train, p3_estimate

ALL_JOBS = p3_train.make_r1_jobs()[44:45]
results_path = os.path.join(RESULTS_DIR, "results.csv")
existing = p3_train.load_existing_result_keys(results_path)

remaining = [j for j in ALL_JOBS
             if not (p3_train._results_key(j.run_id, "on", None, 0) in existing
                     and p3_train._job_grid_is_complete(j, existing))]

print(f"{len(ALL_JOBS)} job(s) selected, {len(remaining)} remaining")
for j in remaining:
    print(" ", j.run_id)

In [ ]:
progress = p3_train.Progress("notebook")
for i, job in enumerate(remaining, 1):
    print(f"\n=== job {i}/{len(remaining)}: {job.run_id} ===")
    t0 = time.time()
    p3_train.run_job(job, DATA_FINAL_ROOT, RESULTS_DIR, progress, run_noise_grid=True)
    print(f"wall time: {time.time()-t0:.1f}s")

In [ ]:
import pandas as pd, subprocess
print(subprocess.run(["du","-sh","/kaggle/working"],capture_output=True,text=True).stdout)
ck = subprocess.run(["find","/kaggle/working","-name","*.safetensors","-o",
                     "-name","pytorch_model.bin"],capture_output=True,text=True).stdout
print("checkpoints left:", ck or "(none)")
df = pd.read_csv(os.path.join(RESULTS_DIR,"results.csv"))
print(df.shape)
print(df.head(10).to_string())
print(df.columns.tolist())

In [ ]:
import subprocess, json as _json
upload_dir = "/kaggle/working/results_upload"
os.makedirs(upload_dir, exist_ok=True)
for f in ("results.csv", "curves.csv"):
    s = os.path.join(RESULTS_DIR, f)
    if os.path.exists(s): shutil.copy(s, os.path.join(upload_dir, f))
pd_ = os.path.join(RESULTS_DIR, "preds")
if os.path.isdir(pd_):
    shutil.copytree(pd_, os.path.join(upload_dir, "preds"), dirs_exist_ok=True)

with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as fh:
    _json.dump({"title": "bangla-noisebench-results",
                "id": "mtutul/bangla-noisebench-results",
                "licenses": [{"name": "unknown"}]}, fh)

out = subprocess.run(["kaggle","datasets","version","-p",upload_dir,
                      "-m",f"session {time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}",
                      "-r","zip","--dir-mode","zip"],
                     capture_output=True, text=True, timeout=900)
print(out.stdout or out.stderr)